Classify images into 10 categories:
airplane, car, bird, cat, deer, dog, frog, horse, ship, truck"

In [1]:
import tensorflow as tf
from tensorflow.keras import layers, models

c:\Users\pc\miniconda3\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.0.1)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(
c:\Users\pc\miniconda3\Lib\site-packages\keras\src\export\tf2onnx_lib.py:8: FutureWarning: In the future `np.object` will be defined as the corresponding NumPy scalar.
  if not hasattr(np, "object"):


In [2]:
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.cifar10.load_data()

c:\Users\pc\miniconda3\Lib\site-packages\keras\src\datasets\cifar.py:18: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  d = cPickle.load(f, encoding="bytes")


Each image is 32×32×3

32 → width

32 → height

3 → RGB (Red, Green, Blue)

Labels are numbers from 0 to 9

In [3]:
x_train = x_train / 255.0
x_test = x_test / 255.0

Original pixel values: 0 – 255

After dividing: 0 – 1

Helps:

Faster learning

Stable gradients

Better accuracy

👉 CNNs always prefer small, consistent values.

In [4]:
model = models.Sequential([
    
    # Convolution Block 1
    layers.Conv2D(32, (3,3), activation='relu', input_shape=(32,32,3)),
    #32 filters → model learns 32 different features
    #3×3 filter size → looks at small image regions
    #Slides over image → detects edges, colors, textures
    #Same filter used everywhere → weight sharing
    #ReLU → removes useless negative values
    #This is where CNN actually “sees” the image.

    layers.MaxPooling2D((2,2)),
    #Takes maximum value from each 2×2 region
    #Reduces image size
    #Keeps strongest features
    #Like summarizing important information.
    
    # Convolution Block 2
    layers.Conv2D(64, (3,3), activation='relu'),
    layers.MaxPooling2D((2,2)),
    
    # Flatten and Dense Layers
    layers.Flatten(),
    #Converts feature maps into 1D vector
    layers.Dense(64, activation='relu'),
    layers.Dense(10, activation='softmax')
])


c:\Users\pc\miniconda3\Lib\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [5]:
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

In [6]:
model.fit(x_train, y_train, epochs=10, batch_size=64)

Epoch 1/10
782/782 ━━━━━━━━━━━━━━━━━━━━ 33s 34ms/step - accuracy: 0.4437 - loss: 1.5469
Epoch 2/10
782/782 ━━━━━━━━━━━━━━━━━━━━ 27s 34ms/step - accuracy: 0.5827 - loss: 1.1891
Epoch 3/10
782/782 ━━━━━━━━━━━━━━━━━━━━ 56s 71ms/step - accuracy: 0.6334 - loss: 1.0503
Epoch 4/10
782/782 ━━━━━━━━━━━━━━━━━━━━ 29s 38ms/step - accuracy: 0.6644 - loss: 0.9691
Epoch 5/10
782/782 ━━━━━━━━━━━━━━━━━━━━ 30s 38ms/step - accuracy: 0.6845 - loss: 0.9119
Epoch 6/10
782/782 ━━━━━━━━━━━━━━━━━━━━ 30s 38ms/step - accuracy: 0.7014 - loss: 0.8615
Epoch 7/10
782/782 ━━━━━━━━━━━━━━━━━━━━ 30s 39ms/step - accuracy: 0.7167 - loss: 0.8173
Epoch 8/10
782/782 ━━━━━━━━━━━━━━━━━━━━ 37s 34ms/step - accuracy: 0.7304 - loss: 0.7769
Epoch 9/10
782/782 ━━━━━━━━━━━━━━━━━━━━ 41s 34ms/step - accuracy: 0.7429 - loss: 0.7423
Epoch 10/10
782/782 ━━━━━━━━━━━━━━━━━━━━ 41s 35ms/step - accuracy: 0.7561 - loss: 0.7062


In [7]:
test_loss, test_acc = model.evaluate(x_test, y_test)
print("Test Accuracy:", test_acc)


313/313 ━━━━━━━━━━━━━━━━━━━━ 4s 10ms/step - accuracy: 0.6881 - loss: 0.9229
Test Accuracy: 0.6880999803543091


✅ UPGRADE 1: Make CNN Slightly Deeper

CIFAR-10 images are complex:

colors

textures

overlapping objects

2 Conv layers are not enough to capture this.

So we:

add one more Conv layer

increase feature extraction depth

In [8]:
model = models.Sequential([

    # Block 1
    layers.Conv2D(32, (3,3), activation='relu', padding='same', input_shape=(32,32,3)),
    layers.MaxPooling2D((2,2)),

    # Block 2
    layers.Conv2D(64, (3,3), activation='relu', padding='same'),
    layers.MaxPooling2D((2,2)),

    # Block 3 (NEW)
    layers.Conv2D(128, (3,3), activation='relu', padding='same'),
    layers.MaxPooling2D((2,2)),

    layers.Flatten(),
    layers.Dense(128, activation='relu'),
    layers.Dense(10, activation='softmax')
])


✅ UPGRADE 2: Add Batch Normalization 
During training:

feature values keep changing

learning becomes unstable

Batch Normalization:

keeps values in a stable range

allows faster & deeper learning

reduces vanishing/exploding gradients

👉 Almost all modern CNNs use it.

In [9]:
model = models.Sequential([

    layers.Conv2D(32, (3,3), padding='same', input_shape=(32,32,3)),
    layers.BatchNormalization(),
    layers.ReLU(),
    layers.MaxPooling2D((2,2)),

    layers.Conv2D(64, (3,3), padding='same'),
    layers.BatchNormalization(),
    layers.ReLU(),
    layers.MaxPooling2D((2,2)),

    layers.Conv2D(128, (3,3), padding='same'),
    layers.BatchNormalization(),
    layers.ReLU(),
    layers.MaxPooling2D((2,2)),

    layers.Flatten(),
    layers.Dense(128, activation='relu'),
    layers.Dense(10, activation='softmax')
])


✅ UPGRADE 3: Add Dropout + Train Longer (Generalization)

CNN is now powerful → risk of overfitting

Dropout:

forces model to not depend on single neurons

improves test accuracy

Also:

CIFAR-10 needs more epochs

In [10]:
model = models.Sequential([

    layers.Conv2D(32, (3,3), padding='same', input_shape=(32,32,3)),
    layers.BatchNormalization(),
    layers.ReLU(),
    layers.MaxPooling2D((2,2)),

    layers.Conv2D(64, (3,3), padding='same'),
    layers.BatchNormalization(),
    layers.ReLU(),
    layers.MaxPooling2D((2,2)),

    layers.Conv2D(128, (3,3), padding='same'),
    layers.BatchNormalization(),
    layers.ReLU(),
    layers.MaxPooling2D((2,2)),

    layers.Flatten(),
    layers.Dense(256, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(10, activation='softmax')
])


In [11]:
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

model.fit(
    x_train, y_train,
    epochs=30,
    batch_size=64,
    validation_data=(x_test, y_test)
)


Epoch 1/30
782/782 ━━━━━━━━━━━━━━━━━━━━ 171s 199ms/step - accuracy: 0.4208 - loss: 1.5930 - val_accuracy: 0.5729 - val_loss: 1.1917
Epoch 2/30
782/782 ━━━━━━━━━━━━━━━━━━━━ 145s 186ms/step - accuracy: 0.5549 - loss: 1.2335 - val_accuracy: 0.5542 - val_loss: 1.2880
Epoch 3/30
782/782 ━━━━━━━━━━━━━━━━━━━━ 195s 177ms/step - accuracy: 0.6121 - loss: 1.0952 - val_accuracy: 0.4922 - val_loss: 1.5655
Epoch 4/30
782/782 ━━━━━━━━━━━━━━━━━━━━ 135s 168ms/step - accuracy: 0.6471 - loss: 1.0048 - val_accuracy: 0.6655 - val_loss: 0.9691
Epoch 5/30
782/782 ━━━━━━━━━━━━━━━━━━━━ 124s 159ms/step - accuracy: 0.6756 - loss: 0.9291 - val_accuracy: 0.6809 - val_loss: 0.9145
Epoch 6/30
782/782 ━━━━━━━━━━━━━━━━━━━━ 123s 157ms/step - accuracy: 0.6920 - loss: 0.8687 - val_accuracy: 0.6912 - val_loss: 0.9079
Epoch 7/30
782/782 ━━━━━━━━━━━━━━━━━━━━ 123s 158ms/step - accuracy: 0.7134 - loss: 0.8195 - val_accuracy: 0.6359 - val_loss: 1.0888
Epoch 8/30
782/782 ━━━━━━━━━━━━━━━━━━━━ 142s 157ms/step - accuracy: 0.7288 -

In [12]:
test_loss, test_acc = model.evaluate(x_test, y_test)
print("Test Accuracy:", test_acc)


313/313 ━━━━━━━━━━━━━━━━━━━━ 5s 15ms/step - accuracy: 0.7529 - loss: 0.9862
Test Accuracy: 0.7529000043869019
